In [ ]:
from datetime import date

import pandas as pd
import numpy as np
import holidays
import yaml


Para inicializar la dimension creamos un dataframe donde vamos a añadir las fechas y demas campos.

Rango ajustado para cubrir los datos reales de `mensajeria_servicio` (2023-09-19 a 2024-08-31), con margen hacia atras y adelante.

In [ ]:
dim_fecha = pd.DataFrame({
    "date": pd.date_range(start='1/1/2023', end='12/31/2026', freq='D')
})
dim_fecha.head()

In [ ]:
dim_fecha["year"] = dim_fecha["date"].dt.year
dim_fecha["month"] = dim_fecha["date"].dt.month
dim_fecha["day"] = dim_fecha["date"].dt.day
dim_fecha["weekday"] = dim_fecha["date"].dt.weekday
dim_fecha["quarter"] = dim_fecha["date"].dt.quarter
dim_fecha["day_of_year"] = dim_fecha["date"].dt.day_of_year
dim_fecha["day_of_month"] = dim_fecha["date"].dt.days_in_month
# NOTA: para que salga en español (Julio, Martes) se necesita el locale 'es_CO.UTF8' instalado (correr 'locale -a' para verificar).
# Si no esta disponible, queda en ingles (July, Tuesday) y se puede traducir despues con un diccionario.
dim_fecha["month_str"] = dim_fecha["date"].dt.month_name()
dim_fecha["day_str"] = dim_fecha["date"].dt.day_name()
dim_fecha["date_str"] = dim_fecha["date"].dt.strftime("%d/%m/%Y")
dim_fecha.head()

# holidays y weekend

In [ ]:
co_holidays = holidays.CO(language="es")
dim_fecha["is_Holiday"] = dim_fecha["date"].apply(lambda x: x in co_holidays)
dim_fecha["holiday"] = dim_fecha["date"].apply(lambda x: co_holidays.get(x))
dim_fecha["saved"] = date.today()
dim_fecha["weekend"] = dim_fecha["weekday"].apply(lambda x: x > 4)
dim_fecha.head()

# Alinear nombres de columnas al diagrama dimensional (version mas actualizada)
ID_Fecha (PK) -> indice al cargar | Fecha_Completa | Ano | Mes | Nombre_Mes | Dia | Nombre_Dia

Las demas columnas (weekday, quarter, day_of_year, day_of_month, date_str, is_Holiday, holiday, weekend, saved) se conservan como atributos extra, no estan en el diagrama pero no estorban.

In [ ]:
dim_fecha.rename(columns={
    'date': 'fecha_completa',
    'year': 'ano',
    'month': 'mes',
    'month_str': 'nombre_mes',
    'day': 'dia',
    'day_str': 'nombre_dia',
}, inplace=True)
dim_fecha.head()

# database connection

In [ ]:
from sqlalchemy import create_engine

with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_etl = config['ETL_PRO']

url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
etl_conn = create_engine(url_etl)

# load

In [ ]:
# NOTA: cambiado de if_exists='replace' a 'append' porque la tabla ahora se crea por DDL
# (ver sqlscripts.yml) con su PK real. 'replace' borraria esa tabla y la reemplazaria sin
# restricciones. Antes de correr este notebook, asegurate que main.py ya ejecuto el DDL
# (o ejecuta el script de sqlscripts.yml manualmente si la tabla aun no existe).
dim_fecha.to_sql('dim_fecha', etl_conn, if_exists='append', index=False)